# 02 · LLM labelling of radiology reports (GPU)
Turns the 4,349 free-text, multilingual radiology reports into 12 soft targets per study.

* Open-weights LLM (Qwen2.5 7B Instruct) run **locally on Kaggle**: no report leaves the platform.
* The model answers one of five categories per finding (definite, probable, possible, negative, not mentioned), mapped to soft targets.
* Prompt with a multilingual glossary; tolerant JSON parser; memory-safe batching sized by token count, with automatic batch splitting on out-of-memory.
* Can resume from a previous `llm_raw.csv` attached as input.
* Validated on the 58 gold studies before use.

Settings: **Internet ON**, accelerator **GPU T4 x2**. Inputs: competition data (+ optional previous output to resume).
Outputs: `llm_raw.csv`, `soft_labels.csv`.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation
import re, json, time, glob, shutil, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import roc_auc_score

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
KEYS = [t.replace("Baker's", "Bakers") for t in TARGETS]
CATS = ["definite", "probable", "possible", "negative", "not_mentioned"]

def walk_inputs(max_depth=5):
    base = "/kaggle/input"
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        dirs[:] = [d for d in dirs if d != "competitions" and "rsna-knee" not in d
                   and not d.endswith("_series") and depth < max_depth]
        yield root, dirs, files

def find(name):
    for root, dirs, files in walk_inputs():
        if name in files:
            return os.path.join(root, name)
    return None

def find_comp_dir():
    for c in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
              "/kaggle/input/rsna-knee-abnormality-detection"]:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError("Attach the competition data.")

COMP = find_comp_dir()
train = pd.read_csv(f"{COMP}/train.csv")
gold_mask = train[TARGETS].notna().any(axis=1)
gold = train[gold_mask].reset_index(drop=True)
rest = train[~gold_mask].reset_index(drop=True)

prev = find("llm_raw.csv")
OUT = "/kaggle/working/llm_raw.csv"
if prev:
    shutil.copy(prev, OUT)
    done_prev = pd.read_csv(OUT)
    print("resuming from", prev, "->", len(done_prev), "reports already labelled,",
          int(done_prev[TARGETS[0]].isna().sum()), "failures")
else:
    print("WARNING: no previous llm_raw.csv found: all 4,349 reports will be labelled (about 3 h).")
print("gold:", len(gold), " unlabelled:", len(rest))

In [ ]:
SYSTEM = r'''You read knee MRI radiology reports. Reports may be in any language (English, Spanish, Turkish, German, French, Portuguese, etc.).
For each of the 12 findings below, classify what the report says about THIS knee into exactly one category:
"definite"  : clearly stated as present
"probable"  : likely, suspected, suggestive of, compatible with
"possible"  : cannot be excluded, questionable, minimal/borderline, to correlate clinically
"negative"  : explicitly stated as absent, normal or intact
"not_mentioned" : the report says nothing about it

Findings:
ACL: tear (partial or complete) or injury of the anterior cruciate ligament or of an ACL graft. Intact graft = negative.
MCL: sprain, tear or injury of the medial collateral ligament.
Medial Meniscus: TEAR of the medial meniscus. Grade I or II intrasubstance degeneration without surface extension is NOT a tear (negative).
Lateral Meniscus: TEAR of the lateral meniscus. Same rule as above.
Medial OA: osteoarthritis of the medial tibiofemoral compartment (cartilage loss, osteophytes, joint space narrowing, subchondral changes). Meniscal degeneration alone is NOT osteoarthritis.
Lateral OA: osteoarthritis of the lateral tibiofemoral compartment. Same rule.
PF OA: patellofemoral osteoarthritis (patellar or trochlear cartilage loss, chondromalacia, osteophytes).
Effusion: joint effusion. Physiological or trace fluid = possible.
Synovitis: synovitis, synovial thickening or inflammation.
Bakers: Baker's cyst or popliteal cyst.
Contusion: bone contusion, bone bruise, traumatic bone marrow edema.
Fracture: any fracture (including stress, insufficiency, avulsion, osteochondral).

Be careful with laterality: medial = internal, lateral = external.
Answer ONLY with a JSON object with exactly these 12 keys:
{"ACL":"...","MCL":"...","Medial Meniscus":"...","Lateral Meniscus":"...","Medial OA":"...","Lateral OA":"...","PF OA":"...","Effusion":"...","Synovitis":"...","Bakers":"...","Contusion":"...","Fracture":"..."}'''

GLOSSARY = """
Multilingual abbreviations and terms:
LCA (Spanish, French, Italian, Portuguese) = ACL. LCP = PCL, which is NOT the ACL.
LCM, LLI, LCI = MCL. LCL, LLE, FCL = lateral collateral ligament, which is NOT the MCL.
menisco interno, ménisque interne, Innenmeniskus, iç menisküs = medial meniscus. externo, externe, Außenmeniskus, dış menisküs = lateral meniscus.
rotura, rupture, déchirure, Riss, yırtık = tear. ön çapraz bağ = ACL. arka çapraz bağ = PCL.
derrame, épanchement, Erguss, efüzyon, sıvı artışı = effusion. edema óseo, œdème osseux, Knochenmarködem, kemik iliği ödemi, contusión = contusion.
fractura, fracture, Fraktur, kırık = fracture. quiste de Baker, kyste poplité, Baker kisti = Baker's cyst.
If the report has an impression or conclusion section, rely on it when the findings are ambiguous."""
RULE = '\nEach value MUST be exactly one of: "definite", "probable", "possible", "negative", "not_mentioned". Never write the name of the finding as a value.'
SYSTEM = SYSTEM + "\n" + GLOSSARY + RULE
print(len(SYSTEM), "characters in the system prompt")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
local = []
for top in os.listdir("/kaggle/input"):
    if "rsna" in top or top == "competitions":
        continue
    local += [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/{top}/**/config.json", recursive=True)
              if "qwen" in p.lower() and "7b" in p.lower() and "instruct" in p.lower()]
path = local[0] if local else MODEL_ID
print("model:", path)
tok = AutoTokenizer.from_pretrained(path, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map="auto")
model.eval()
print(torch.cuda.device_count(), "GPU")

In [ ]:
MAX_REPORT_TOKENS = 2500
TOKEN_BUDGET = 14000      # prompt tokens per batch (batch size x longest prompt)
MAX_BATCH = 16

def build_prompt(report):
    ids = tok(str(report), add_special_tokens=False)["input_ids"][:MAX_REPORT_TOKENS]
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "Report:\n" + tok.decode(ids)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# ---- parser: same as the validated K2 run (fixes 1 and 3) ----
SYN = {
    "definite": "definite", "present": "definite", "yes": "definite", "positive": "definite", "confirmed": "definite",
    "probable": "probable", "likely": "probable", "suspected": "probable", "suggestive": "probable",
    "possible": "possible", "questionable": "possible", "equivocal": "possible", "uncertain": "possible", "indeterminate": "possible",
    "negative": "negative", "absent": "negative", "no": "negative", "normal": "negative", "intact": "negative", "none": "negative",
    "not_mentioned": "not_mentioned", "notmentioned": "not_mentioned", "unknown": "not_mentioned", "na": "not_mentioned", "n/a": "not_mentioned",
}
NEG_WORDS = ("no ", "not ", "without", "intact", "normal", "absent", "none", "negative", "unremarkable")

def norm_cat(v):
    v = str(v).strip().lower()
    if not v:
        return None
    for cand in (v, v.replace(" ", "_"), v.replace("-", "_"), v.split()[0].strip(",.;:")):
        if cand in SYN:
            return SYN[cand]
    if any(w in v + " " for w in NEG_WORDS):
        return "negative"
    return "definite"

def parse(text):
    if "{" not in text:
        return None
    blob = text[text.index("{"):]
    blob = blob[:blob.rindex("}") + 1] if "}" in blob else blob + "}"
    try:
        d = json.loads(blob)
    except Exception:
        d = dict(re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', blob))
    if not d:
        return None
    norm = {re.sub(r"[^a-z]", "", str(k).lower()): v for k, v in d.items()}
    out, hits = [], 0
    for k in KEYS:
        v = norm.get(re.sub(r"[^a-z]", "", k.lower()))
        c = norm_cat(v) if v is not None else None
        hits += c is not None
        out.append(c or "not_mentioned")
    return out if hits >= 6 else None

# ---- memory-safe generation ----
@torch.no_grad()
def generate(prompts, max_new_tokens=200):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def generate_safe(prompts, max_new_tokens=200):
    """On out-of-memory, free the cache and split the batch in two (down to one report)."""
    try:
        return generate(prompts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            print("  out of memory on a single report: it is left unlabelled")
            return [""]
        h = len(prompts) // 2
        return generate_safe(prompts[:h], max_new_tokens) + generate_safe(prompts[h:], max_new_tokens)

def make_batches(lengths):
    """Group prompts (sorted by length) so that batch size x longest prompt stays under the budget."""
    batches, cur, cur_max = [], [], 0
    for i, L in lengths:
        new_max = max(cur_max, L)
        if cur and (len(cur) + 1 > MAX_BATCH or (len(cur) + 1) * new_max > TOKEN_BUDGET):
            batches.append(cur); cur, new_max = [], L
        cur.append(i); cur_max = new_max
    if cur:
        batches.append(cur)
    return batches

def label_frame(df, out_csv, save_every=10):
    done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["StudyInstanceUID"] + TARGETS)
    todo = df[~df["StudyInstanceUID"].isin(done["StudyInstanceUID"])].reset_index(drop=True)
    prompts = [build_prompt(r) for r in todo["Report"]]
    lengths = sorted(((i, len(tok(p)["input_ids"])) for i, p in enumerate(prompts)), key=lambda x: x[1])
    batches = make_batches(lengths)
    print(len(todo), "reports to label in", len(batches), "batches; longest prompt",
          max([L for _, L in lengths], default=0), "tokens")
    rows, t0, n = [], time.time(), 0
    for b, idx in enumerate(batches):
        texts = generate_safe([prompts[i] for i in idx])
        for i, txt in zip(idx, texts):
            cats = parse(txt)
            if cats is None:
                cats = parse(generate_safe([prompts[i]], max_new_tokens=400)[0])
            rows.append([todo.at[i, "StudyInstanceUID"]] + (cats if cats else [np.nan] * 12))
        n += len(idx)
        if (b + 1) % save_every == 0 or b == len(batches) - 1:
            done = pd.concat([done, pd.DataFrame(rows, columns=["StudyInstanceUID"] + TARGETS)], ignore_index=True)
            done.to_csv(out_csv, index=False); rows = []
            print(f"{n}/{len(todo)}  {time.time()-t0:.0f} s  failures so far: {int(done[TARGETS[0]].isna().sum())}")
    return done

SOFT = {"definite": 0.95, "probable": 0.80, "possible": 0.40, "negative": 0.03}
NOT_MENTIONED = 0.07

def to_soft(raw):
    out = raw[["StudyInstanceUID"]].copy()
    for t in TARGETS:
        out[t] = raw[t].map(SOFT).fillna(NOT_MENTIONED).astype(float)
    return out

In [ ]:
# ---- Resume the full run ----
raw_rest = label_frame(rest, OUT)
raw_rest = raw_rest.drop_duplicates("StudyInstanceUID", keep="last")
raw_rest = raw_rest[raw_rest["StudyInstanceUID"].isin(rest["StudyInstanceUID"])]
soft = to_soft(raw_rest)
soft.to_csv("/kaggle/working/soft_labels.csv", index=False)
print(len(soft), "studies in soft_labels.csv (expected 4349)")
print("failures:", int(raw_rest[TARGETS[0]].isna().sum()))
print(pd.DataFrame({"LLM soft mean": soft[TARGETS].mean(), "gold prevalence": gold[TARGETS].fillna(0).mean()}).round(3))

In [ ]:
# ---- Same pipeline on the 58 gold studies (sanity check, about 3 min) ----
raw_gold = label_frame(gold, "/kaggle/working/llm_raw_gold.csv")
s = to_soft(raw_gold).set_index("StudyInstanceUID").loc[gold["StudyInstanceUID"]]
aucs = {t: roc_auc_score(gold[t].fillna(0), s[t]) for t in TARGETS}
print(pd.Series(aucs).round(3)); print("Gold macro AUC:", round(float(np.mean(list(aucs.values()))), 4))